In [1]:
# Set working directory to base directory for all notebooks in subdirectories
import os
from pathlib import Path
path = Path(os.getcwd())

# update base working directory to QFlow-autoannotator
if path.stem != 'falcon-core':
    base_dir = path.parents[0]
    os.chdir(base_dir)
else:
    base_dir = path

cpp_root_dir = os.path.join(base_dir, "cpp")
capi_root_dir = os.path.join(base_dir, "c-api")
metadata_root_dir = os.path.join(base_dir, "code_docs", "capi_docs")
script_dir = os.path.join(metadata_root_dir, "scripts")
print(f"Base directory for all modules: {base_dir}")
print(f"Directory for all c-api documentation scripts: {script_dir}")
print(f"metadata dir: {metadata_root_dir}")
print(f"C++ header dir: {cpp_root_dir}")
print(f"Directory for all c-api documentation scripts: {capi_root_dir}")

Base directory for all modules: /home/zach/Documents/github/FALCon/falcon-core/code_docs
Directory for all c-api documentation scripts: /home/zach/Documents/github/FALCon/falcon-core/code_docs/code_docs/capi_docs/scripts
metadata dir: /home/zach/Documents/github/FALCon/falcon-core/code_docs/code_docs/capi_docs
C++ header dir: /home/zach/Documents/github/FALCon/falcon-core/code_docs/cpp
Directory for all c-api documentation scripts: /home/zach/Documents/github/FALCon/falcon-core/code_docs/c-api


# Port Doxygen comment blocks to program specific documentation format

The goal of this notebook is to enable user to rapidly generate documentation for programming languages which effectively "wrap" the falcon-core C++ code base. This is necessary for creating auto-tunning algorithmic builds on-top-of a desired programming language.

Current wrapper support is for:
- C 

For demonstration purposes, this notebook outlines how to map C++ documentation to C via the following Python scripts: 

1) `/docs_managment_code/upgrade_doxygen_comments.py`: Clean-up comment blocks to ensure doxygen formatting,
2) `/docs_managment_code/extract_cpp_docs.py`: Extract doxygen comment blocks and temporarily store in a cpp_metadata directory,
3)  `/docs_managment_code/generate_c_api_maps.py`: Construct cpp code mapping configuration yaml files for each cpp file,
4) `/docs_managment_code/inject_c_docs.py`: Inject cpp comment blocks in desired programming language code,
5) Helper scripts.


However, the make file `capi_docs.mk` contains all of the necessary functionality for maintaining the c-api documentation and has been included in the primary `Makefile`.

Commands:`make docs-all`, `docs-setup`, `docs-run`, `docs-coverage`, `docs-teardown`, `clean_logs`

`make docs-all`: docs-setup docs-run docs-coverage


## 1. Clean C++ Comment Blocks

This script updates any `/* ... strings ... */` to `/** ... strings ... */`, which are Doxygen formatted C++ comment blocks.

In [ ]:
!python3 {script_dir}/upgrade_doxygen_comments.py ./cpp/include --write --verbose

**NOTE:** The comment style assumes Doxygen formats for extraction and expects the following styles for injected documentation:

```c
// @category:<category>
/* AUTO-DOC from cpp: <c function> | <namespace of cpp function> */
/**
 * Doxygen comment block generated from auto-linked cpp-capi code
 */

/* MAN-DOC from cpp: <c function> | <namespace of cpp function> */
/**
 * Doxygen comment block generated from manually-linked cpp<->c-api code
 */

/* USER-DOC */
/**
 * Doxygen comment block generated inline in c-api code by user.
 */
```

### 1.1 Merging User and Auto/Man Doxygen Comments

We use a toy example where we assume that Vector_c_api.h has a function that reads like:

```c
// @category:read
/* USER-DOC */
/**
 * @brief Return the end point of a vector in device coordinates.
 *
 * This function is part of the public C API. It returns the end point
 * associated with the given vector handle, after applying the current
 * device transform.
 *
 * @param v  Handle to a valid Vector instance.
 * @return   A handle to the end point Vector.
 */
PointHandle Vector_end_point(VectorHandle handle);
```

and a Vectort.hpp header that reads like:

```c
  /**
   * @brief Returns the point at the end.
   */
  const PointSP endPoint() const;
```

The injector script treats user documentation as the priority when merging documentation. In this example, the injector script will generate a new doxygen comment like::

```c

```

## 2. Extract C++ Comment Blocks to Metadata Directory

This script scrapes the desired target directory of all Doxygen formatted strings and dumps them into a library for future reference.

**NOTE:** The is no inherent reason to save this library, but it is instructive to have for debugging purposes.

In [ ]:
# 1) Extract metadata into docs_managment_code/cpp_metadata
!python3 {script_dir}/extract_cpp_docs.py \
  ./cpp/include \
  --meta-root ./{metadata_root_dir}/cpp_metadata

## 3. Generate Configuration Files that Map C++ Documentation to the Desired Programming Language

This script automates the write of .map.yml files into the target language directory (in this example C code). Current expectation is that the C code maps to C++ code via `<C++ Class Name>_<Class methhod>`.

In [ ]:
# 2) Generate auto maps
!python3 {script_dir}/generate_c_api_maps.py \
  --cpp-metadata-root {metadata_root_dir}/ \
  --cpp-include-root  {cpp_root_dir}/include/ \
  --c-api-root        {capi_root_dir}/include/ \
  --overwrite \
  --verbose

## 4. Inject Doxygen Formatted Documentation

This script inserts Doxygen comment blocks into the desired language, given a .map.yml mapping.

**NOTE:** This current script assumes Doxygen injection to C code. If a different language is desired, then a new injector must be created.

In [ ]:
print(f'{metadata_root_dir}/cpp_metadata')

In [ ]:
!python3 {script_dir}/inject_c_docs.py \
  --capi-root {capi_root_dir} \
  --cpp-root {cpp_root_dir} \
  --cpp-metadata-root {metadata_root_dir}/cpp_metadata \
  --maps-dir {capi_root_dir}/include/falcon_core \
  --user-maps-dir {metadata_root_dir}/c-api_user_maps \
  --out-root {capi_root_dir} \
  --verbose


## 5. Helper Scripts

This script compares the coverage between C++ and the desired language.

**NOTE:** This assumes the target language is C.

In [48]:
!python3 {script_dir}/doxygen_port_coverage.py \
    --cpp-root           {cpp_root_dir}/include \
    --cpp-metadata-root  {cpp_root_dir}/cpp_metadata \
    --c-root             {capi_root_dir}/include

C++ headers root: /home/zach/Documents/github/FALCon/falcon-core/cpp/include
C  headers root:  /home/zach/Documents/github/FALCon/falcon-core/c-api/include

C header documentation coverage (local):
  Functions with Doxygen: 1/2530 (0.0%)

C++ header documentation coverage (local, scanned via extract_cpp_docs):
  Functions with Doxygen: 364/533

Port coverage (C docs vs C++ docs):
  C++ functions with Doxygen (from headers): 551
  Ported docs (C):           0/364 (0.0%)

Logs written:
  - cpp_scanned_doc_functions.log
  - cpp_metadata_doc_functions.log
  - cpp_ported_functions.log
  - cpp_left_to_port_functions.log
